# LAB1 — Phần V, yêu cầu 2: Khảo sát và tiền xử lý dữ liệu Bank Marketing

**Sinh viên:** TRẦN TRUNG THÔNG  
**MSSV:** 6451071075  
**Môn:** Khai thác dữ liệu

**Yêu cầu tài liệu (PDF trang 28–29):** *"Sinh viên thực hiện khảo sát và tiền xử lý dữ liệu cho dữ liệu Bank marketing giống như phần hướng dẫn."*

- Sử dụng file thầy phát: `bank-20261117T050329Z-1-001.zip`, chứa `bank/bank-full.csv` (**45.211 dòng, 17 cột**).
- Bảng ví dụ PDF có `cons.price.idx` nhưng tập dữ liệu thực tế **không có** cột này; các bước dưới đây sử dụng cột có thật, không giả tạo dữ liệu.
- Các ô code **độc lập với 8 notebook Titanic** nhưng phải chạy **từ trên xuống** trong notebook này.
- Chụp ảnh kết quả **bao gồm thanh taskbar Windows** theo quy định của giảng viên.


## 0. Thông tin sinh viên và chuẩn bị thư viện

In [ ]:
print("Họ và tên: TRẦN TRUNG THÔNG")
print("MSSV: 6451071075")
print("LAB1 - Phần V, Yêu cầu 2 - Bank Marketing")

import io
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', 30)
print('Thư mục làm việc:', Path.cwd())

## 1. Tải bộ dữ liệu do thầy cung cấp

**Trên Colab:** Bấm biểu tượng 📁 Tệp bên trái → biểu tượng ⬆ Tải tệp lên → chọn file ZIP **`bank-20261117T050329Z-1-001.zip`** (không cần giải nén). Chạy ô dưới sau khi tải lên. Nếu bạn đã giải nén, có thể upload `bank-full.csv` thay thế.

**Quan trọng:** File CSV phân cách bằng dấu `;` (không phải dấu phẩy).

In [ ]:
working_dir = Path.cwd()
zip_files = sorted(working_dir.glob('bank*.zip'))
csv_files = list(working_dir.glob('bank-full.csv')) + list(working_dir.glob('bank/bank-full.csv'))

if zip_files:
    source = zip_files[0]
    with zipfile.ZipFile(source) as zipped:
        file_inside = next((f for f in zipped.namelist() if f.endswith('bank-full.csv')), None)
        if file_inside is None:
            raise FileNotFoundError('File ZIP không chứa bank-full.csv')
        df = pd.read_csv(io.BytesIO(zipped.read(file_inside)), sep=';')
    print('Đã đọc dữ liệu từ ZIP:', source.name, '→', file_inside)
elif csv_files:
    source = csv_files[0]
    df = pd.read_csv(source, sep=';')
    print('Đã đọc dữ liệu CSV:', source)
else:
    raise FileNotFoundError('Chưa tìm thấy file bank*.zip hoặc bank-full.csv. Upload dữ liệu vào mục Tệp ở Colab rồi chạy lại.')

print('Kích thước dữ liệu:', df.shape)
print('Tên các cột:', df.columns.tolist())
display(df.head(5))
assert df.shape[1] == 17, 'Dữ liệu phải có 17 cột: kiểm tra lại sep và file.'
assert 'y' in df.columns, 'Thiếu thuộc tính mục tiêu y.' 

### Giải thích thuộc tính quan trọng

- `job`: nghề nghiệp; `marital`: tình trạng hôn nhân; `education`: trình độ học vấn.
- `default`: có nợ tín dụng mặc định hay không; `housing`: vay nhà; `loan`: vay cá nhân.
- `duration`: thời lượng cuộc gọi cuối cùng (giây); `balance`: số dư tài khoản; `age`: tuổi.
- `y`: khách hàng có đăng ký tiền gửi có kỳ hạn (`yes`) hay không (`no`).

**Chú ý:** File thực tế dùng `marital`, không phải cách viết `Marial` trong bảng minh họa PDF; cũng không chứa `cons.price.idx`.


## 2. Khảo sát cấu trúc và thống kê mô tả

In [ ]:
print('Số hàng, số cột:', df.shape)
print('\nThông tin từng cột:')
df.info()
print('\nThống kê các cột số:')
display(df.describe().T.round(2))
print('\nPhân bố đăng ký gửi tiền (y):')
display(pd.DataFrame({'so_luong': df['y'].value_counts(), 'ty_le_%': (df['y'].value_counts(normalize=True)*100).round(2)}))
print('\nSố lượng theo nghề nghiệp:')
display(df['job'].value_counts().to_frame('so_luong'))

**Nhận xét gợi ý:** Ghi số lượng bản ghi, số thuộc tính, tuổi trung bình, tỷ lệ khách hàng đăng ký tiền gửi (cột `y`). Dữ liệu mất cân bằng giữa hai nhóm `yes`/`no`, nên khi đánh giá không chỉ nhìn số lượng tuyệt đối.


## 3. Kiểm tra dữ liệu thiếu và giá trị `unknown`

In [ ]:
missing = pd.DataFrame({
    'so_gia_tri_NaN': df.isna().sum(),
    'ty_le_NaN_%': (df.isna().mean()*100).round(2)
}).sort_values('so_gia_tri_NaN', ascending=False)
print('Giá trị thiếu thật sự (NaN):')
display(missing)

category_cols = df.select_dtypes(include='object').columns.tolist()
unknown = pd.Series({col: (df[col].astype(str).str.lower()=='unknown').sum() for col in category_cols}).sort_values(ascending=False)
print('Số lần xuất hiện chuỗi unknown (KHÁC với NaN):')
display(unknown.to_frame('so_luong_unknown'))
print('Dòng trùng lặp:', df.duplicated().sum())

**Diễn giải:** Dữ liệu không có `NaN` không đồng nghĩa mọi thông tin đều đã biết: một số cột chứa chữ `unknown`. Không nên tự động gộp `poutcome='unknown'` với thiếu dữ liệu; giá trị này còn có ý nghĩa nghiệp vụ là kết quả chiến dịch trước không xác định. Dòng trùng lặp cũng có thể là các khách hàng khác nhau trùng thông tin, nên **không xóa hàng loạt** nếu chưa có khóa định danh.


## 4. Tiền xử lý dữ liệu thiếu/không rõ

In [ ]:
df_clean = df.copy()
# Chỉ biến unknown thành dữ liệu thiếu ở các cột mô tả thông tin chưa rõ.
cols_to_impute = ['job', 'education', 'contact']
for col in cols_to_impute:
    df_clean[col] = df_clean[col].replace('unknown', np.nan)
    df_clean[col] = df_clean[col].fillna('Khong_ro')

# Giữ nguyên poutcome='unknown' do đây có thể là trạng thái nghiệp vụ có ý nghĩa.
# Nếu phát sinh số thiếu thật ở các cột số, điền bằng trung vị.
for col in df_clean.select_dtypes(include='number').columns:
    if df_clean[col].isna().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print('Số NaN còn lại:', int(df_clean.isna().sum().sum()))
print('Ví dụ cột education sau xử lý:')
display(df_clean['education'].value_counts().head(8).to_frame('so_luong'))

## 5. Binning và Smoothing (làm trơn tuổi theo nhóm)

In [ ]:
df_age = df_clean[['age']].copy()
df_age['age_bin'] = pd.cut(df_age['age'], bins=[0,20,30,40,50,60,100], include_lowest=True)
df_age['age_smoothed'] = df_age.groupby('age_bin', observed=True)['age'].transform('mean')
print('Tuổi gốc và tuổi sau làm trơn:')
display(df_age.head(12))
print('Thống kê theo nhóm:')
display(df_age.groupby('age_bin', observed=True)['age'].agg(['count','mean','min','max']).round(2))

## 6. Rời rạc hóa tuổi thành nhãn

In [ ]:
df_clean['age_group'] = pd.cut(df_clean['age'],
    bins=[-np.inf, 30, 60, np.inf], labels=['youth', 'adult', 'senior'])
print('Quy tắc minh họa: youth ≤30; adult >30 đến ≤60; senior >60')
display(df_clean[['age', 'age_group']].head(10))
display(df_clean['age_group'].value_counts().sort_index().to_frame('so_khach_hang'))

## 7. Mã hóa One-hot Encoding các biến phân loại

In [ ]:
onehot = pd.get_dummies(df_clean[['marital', 'education']], dtype=int)
print('Các cột sau One-hot Encoding:')
print(onehot.columns.tolist())
display(pd.concat([df_clean[['marital','education']].head(10), onehot.head(10)], axis=1))

## 8. Chuẩn hóa Min-max về [0, 1]

In [ ]:
numeric_cols = ['age','balance','duration']
scaled = df_clean[numeric_cols].copy()
for col in numeric_cols:
    lo, hi = scaled[col].min(), scaled[col].max()
    scaled[col+'_minmax'] = (scaled[col] - lo)/(hi-lo) if hi > lo else 0.0
print('So sánh giá trị ban đầu và Min-max:')
display(scaled.head(10).round(4))
for col in numeric_cols:
    print(f'{col}_minmax: min={scaled[col+"_minmax"].min():.1f}, max={scaled[col+"_minmax"].max():.1f}')

## 9. Trực quan hóa dữ liệu Bank Marketing

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,4))
sns.histplot(data=df_clean,x='age',hue='y',multiple='layer',bins=25,ax=axes[0])
axes[0].set_title('Phân bố tuổi theo kết quả đăng ký')
axes[0].set_xlabel('Tuổi')
axes[0].set_ylabel('Số khách hàng')

rate_job = (df_clean.assign(subscribed=(df_clean['y']=='yes').astype(int))
            .groupby('job', observed=True)['subscribed'].agg(['mean','size'])
            .sort_values('mean',ascending=False))
sns.barplot(x=rate_job['mean']*100,y=rate_job.index,ax=axes[1],color='steelblue')
axes[1].set_title('Tỷ lệ đăng ký theo nghề nghiệp')
axes[1].set_xlabel('Tỷ lệ đăng ký (%)')
axes[1].set_ylabel('Nghề nghiệp')
plt.tight_layout()
plt.show()
print('Bảng tỷ lệ và số khách theo nghề nghiệp:')
display(rate_job.rename(columns={'mean':'ty_le_dang_ky','size':'so_khach'}).assign(ty_le_dang_ky=lambda t:(t['ty_le_dang_ky']*100).round(2)))

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
rate_marital=(df_clean.groupby('marital')['y'].apply(lambda s: (s=='yes').mean()*100)).sort_values(ascending=False)
sns.barplot(x=rate_marital.index,y=rate_marital.values,ax=axes[0],color='steelblue')
axes[0].set_title('Tỷ lệ đăng ký theo hôn nhân')
axes[0].set_ylabel('Tỷ lệ đăng ký (%)')
axes[0].set_xlabel('Tình trạng hôn nhân')
rate_housing=(df_clean.groupby('housing')['y'].apply(lambda s: (s=='yes').mean()*100)).sort_index()
sns.barplot(x=rate_housing.index,y=rate_housing.values,ax=axes[1],color='coral')
axes[1].set_title('Tỷ lệ đăng ký theo vay mua nhà')
axes[1].set_ylabel('Tỷ lệ đăng ký (%)')
axes[1].set_xlabel('Có vay mua nhà?')
plt.tight_layout()
plt.show()
print('Bảng đối chiếu kết quả y theo tình trạng hôn nhân:')
display(pd.crosstab(df_clean['marital'],df_clean['y'],margins=True))

### Nhận xét kết quả (hoàn thiện theo bảng/biểu đồ sau khi chạy)

1. Ghi **số khách hàng và tỷ lệ đăng ký tiền gửi** trên toàn bộ dữ liệu.
2. Xác định **nhóm nghề nghiệp có tỷ lệ đăng ký cao nhất/thấp nhất** và so sánh cả `so_khach` để tránh kết luận mạnh từ nhóm nhỏ.
3. Đối chiếu tỷ lệ theo tình trạng hôn nhân và vay mua nhà.
4. Trình bày quá trình tiền xử lý: `NaN` vs. `unknown`, chia nhóm tuổi, làm trơn, One-hot Encoding và Min-max.

**Cảnh báo quan trọng về `duration`:** Thời lượng cuộc gọi cuối cùng chỉ biết sau khi đã gọi. Nếu sau này xây dựng mô hình dự đoán trước khi liên lạc, không nên dùng thuộc tính này vì có nguy cơ rò rỉ thông tin. Trong LAB1 chúng ta chỉ khảo sát và tiền xử lý.


## 10. Lưu dữ liệu đã tiền xử lý (tùy chọn)

In [ ]:
# Giữ file CSV làm sản phẩm trung gian; không bắt buộc nộp nếu thầy chỉ yêu cầu mã nguồn.
output_file = '6451071075_bank_marketing_clean.csv'
df_clean.to_csv(output_file, index=False, encoding='utf-8-sig')
print('Đã xuất:', output_file)
print('Dữ liệu sau xử lý:', df_clean.shape)
print('Họ và tên: TRẦN TRUNG THÔNG | MSSV: 6451071075')

## Checklist nộp bài

- [ ] Đã chạy từ trên xuống và notebook có kết quả/bảng/biểu đồ.
- [ ] Đầu file có `print` họ tên và MSSV; cuối file cũng hiển thị thông tin.
- [ ] Đã bổ sung nhận xét từ kết quả **thực tế**, không dùng số liệu ví dụ để nộp.
- [ ] Đã chụp ảnh kết quả **có thanh taskbar Windows**.
- [ ] Vào **Tệp → Tải xuống → Tải xuống .ipynb**, lưu với tên `6451071075_LAB1_Bai9.ipynb` (tên *Bài 9* là cách đánh số tiếp nối 8 bài Titanic; nếu thầy yêu cầu số thứ tự khác thì đổi tên theo quy định của thầy).
- [ ] Upload file vào repository `6451071075_Lab1`, có thể tạo `screenshots/` để lưu ảnh.

**Lưu ý:** Chỉ gửi lên GitHub **sau khi đã chạy và lưu kết quả**; upload ZIP dữ liệu thầy cung cấp là tùy theo hướng dẫn giảng viên và điều kiện chia sẻ dữ liệu.
